In [1]:
import os 
os.chdir(os.path.join(os.path.abspath("."), ".."))
print(os.getcwd())

/home/harim/Desktop/kaist/DS801/StockMixer


In [2]:
import random
import numpy as np
import torch as torch
from tqdm import tqdm
from src.evaluator import evaluate
from src.model import get_loss, StockMixer
import pickle
from utils import set_seed

In [3]:
device = torch.device("cuda") if torch.cuda.is_available() else 'cpu'
data_path = './dataset'
market_name = 'NASDAQ'

stock_num = 1026
lookback_length = 16
valid_index = 756
test_index = 1008

epochs = 100
fea_num = 5
market_num = 20
steps = 1
learning_rate = 0.001
alpha = 0.1
scale_factor = 3
activation = 'GELU'

seed = 1234

In [4]:
set_seed(seed)
dataset_path = os.path.join(data_path, market_name)
with open(os.path.join(dataset_path, "eod_data.pkl"), "rb") as f:
    eod_data = pickle.load(f)
with open(os.path.join(dataset_path, "mask_data.pkl"), "rb") as f:
    mask_data = pickle.load(f)
with open(os.path.join(dataset_path, "gt_data.pkl"), "rb") as f:
    gt_data = pickle.load(f)
with open(os.path.join(dataset_path, "price_data.pkl"), "rb") as f:
    price_data = pickle.load(f)


In [5]:
def get_batch(offset=None):
    if offset is None:
        offset = random.randrange(0, valid_index)
    seq_len = lookback_length
    mask_batch = mask_data[:, offset: offset + seq_len + steps]
    mask_batch = np.min(mask_batch, axis=1)
    return (
        eod_data[:, offset:offset + seq_len, :],
        np.expand_dims(mask_batch, axis=1),
        np.expand_dims(price_data[:, offset + seq_len - 1], axis=1),
        np.expand_dims(gt_data[:, offset + seq_len + steps - 1], axis=1))

In [6]:
def validate(start_index, end_index):
    with torch.no_grad():
        cur_valid_pred = np.zeros([stock_num, end_index - start_index], dtype=float)
        cur_valid_gt = np.zeros([stock_num, end_index - start_index], dtype=float)
        cur_valid_mask = np.zeros([stock_num, end_index - start_index], dtype=float)
        loss = 0.
        reg_loss = 0.
        rank_loss = 0.
        for cur_offset in range(start_index - lookback_length - steps + 1, end_index - lookback_length - steps + 1):
            data_batch, mask_batch, price_batch, gt_batch = map(

                lambda x: torch.Tensor(x).to(device),
                get_batch(cur_offset)
            )
            prediction = model(data_batch)
            cur_loss, cur_reg_loss, cur_rank_loss, cur_rr = get_loss(prediction, gt_batch, price_batch, mask_batch,
                                                                     stock_num, alpha)
            loss += cur_loss.item()
            reg_loss += cur_reg_loss.item()
            rank_loss += cur_rank_loss.item()
            cur_valid_pred[:, cur_offset - (start_index - lookback_length - steps + 1)] = cur_rr[:, 0].cpu()
            cur_valid_gt[:, cur_offset - (start_index - lookback_length - steps + 1)] = gt_batch[:, 0].cpu()
            cur_valid_mask[:, cur_offset - (start_index - lookback_length - steps + 1)] = mask_batch[:, 0].cpu()
        loss = loss / (end_index - start_index)
        reg_loss = reg_loss / (end_index - start_index)
        rank_loss = rank_loss / (end_index - start_index)
        cur_valid_perf = evaluate(cur_valid_pred, cur_valid_gt, cur_valid_mask)
    return loss, reg_loss, rank_loss, cur_valid_perf


In [7]:
trade_dates = mask_data.shape[1]
model = StockMixer(
    stocks=stock_num,
    time_steps=lookback_length,
    channels=fea_num,
    market=market_num,
    scale=scale_factor
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
best_valid_loss = np.inf
best_test_loss = np.inf
best_valid_perf = None
best_test_perf = None
batch_offsets = np.arange(start=0, stop=valid_index, dtype=int)

In [8]:
for epoch in range(epochs):
    np.random.shuffle(batch_offsets)
    tra_loss = 0.0
    tra_reg_loss = 0.0
    tra_rank_loss = 0.0
    data_counts = valid_index - lookback_length - steps + 1
    
    pbar = tqdm(range(data_counts))
    for j in pbar:
        data_batch, mask_batch, price_batch, gt_batch = map(
            lambda x: torch.Tensor(x).to(device),
            get_batch(batch_offsets[j])
        )
        optimizer.zero_grad()
        prediction = model(data_batch)
        cur_loss, cur_reg_loss, cur_rank_loss, _ = get_loss(prediction, gt_batch, price_batch, mask_batch,
                                                            stock_num, alpha)
        cur_loss.backward()
        optimizer.step()
        
        pbar.set_description(f'Epochs :{epoch}, cur_loss : {cur_loss:.5f},cur_reg_loss : {cur_reg_loss:.5f},cur_rank_loss : {cur_rank_loss:.5f}')

        tra_loss += cur_loss.item()
        tra_reg_loss += cur_reg_loss.item()
        tra_rank_loss += cur_rank_loss.item()

    tra_loss = tra_loss / data_counts
    tra_reg_loss = tra_reg_loss / data_counts
    tra_rank_loss = tra_rank_loss / data_counts
    print(f'Train : loss:{tra_loss:.4f}  =  {tra_reg_loss:.4f} + alpha*{tra_rank_loss:.4f}')

    val_loss, val_reg_loss, val_rank_loss, val_perf = validate(valid_index, test_index)
    print(f'Valid : loss:{val_loss:.4f}  =  {val_reg_loss:.4f} + alpha*{val_rank_loss:.4f}')
    
    test_loss, test_reg_loss, test_rank_loss, test_perf = validate(test_index, trade_dates)
    print(f'Test: loss:{test_loss:.4f}  =  {test_reg_loss:.4f} + alpha*{test_rank_loss:.4f}')

    if test_loss < best_test_loss:
        best_test_loss = test_loss
        best_valid_perf = val_perf
        best_test_perf = test_perf

    print(f'Valid performance: mse:{val_perf["mse"]:.4f} IC:{val_perf["IC"]:.4f} RIC:{val_perf["RIC"]:.4f} prec@10:{val_perf["prec_10"]:.4f}, SR:{val_perf["sharpe5"]:.4f}')
    print(f'BestTest performance: mse:{best_test_perf["mse"]:.4f} IC:{best_test_perf["IC"]:.4f} RIC:{best_test_perf["RIC"]:.4f} prec@10:{best_test_perf["prec_10"]:.4f}, SR:{best_test_perf["sharpe5"]:.4f}\n\n')


Epochs :0, cur_loss : 0.00185,cur_reg_loss : 0.00182,cur_rank_loss : 0.00031: 100%|██████████| 740/740 [00:08<00:00, 85.88it/s] 


Train : loss:0.0125  =  0.0124 + alpha*0.0006
Valid : loss:0.0020  =  0.0020 + alpha*0.0004
Test: loss:0.0016  =  0.0016 + alpha*0.0003
Valid performance: mse:0.0020 IC:0.0177 RIC:0.2050 prec@10:0.5329, SR:1.9389
BestTest performance: mse:0.0016 IC:0.0226 RIC:0.2933 prec@10:0.5207, SR:1.2889




Epochs :1, cur_loss : 0.00074,cur_reg_loss : 0.00072,cur_rank_loss : 0.00016: 100%|██████████| 740/740 [00:07<00:00, 95.64it/s] 


Train : loss:0.0017  =  0.0017 + alpha*0.0003
Valid : loss:0.0016  =  0.0015 + alpha*0.0003
Test: loss:0.0012  =  0.0011 + alpha*0.0002
Valid performance: mse:0.0015 IC:0.0167 RIC:0.1829 prec@10:0.5266, SR:2.0253
BestTest performance: mse:0.0012 IC:0.0266 RIC:0.3392 prec@10:0.5257, SR:1.3664




Epochs :2, cur_loss : 0.00236,cur_reg_loss : 0.00234,cur_rank_loss : 0.00022: 100%|██████████| 740/740 [00:07<00:00, 104.91it/s]


...

Train : loss:0.0006  =  0.0006 + alpha*0.0000
Valid : loss:0.0006  =  0.0006 + alpha*0.0001
Test: loss:0.0005  =  0.0005 + alpha*0.0001
Valid performance: mse:0.0006 IC:0.0324 RIC:0.2730 prec@10:0.5317, SR:1.9288
BestTest performance: mse:0.0004 IC:0.0105 RIC:0.1522 prec@10:0.5287, SR:1.7961




Epochs :33, cur_loss : 0.00028,cur_reg_loss : 0.00027,cur_rank_loss : 0.00003: 100%|██████████| 740/740 [00:09<00:00, 79.89it/s] 


Train : loss:0.0006  =  0.0006 + alpha*0.0000
Valid : loss:0.0006  =  0.0006 + alpha*0.0001
Test: loss:0.0004  =  0.0004 + alpha*0.0000
Valid performance: mse:0.0006 IC:-0.0098 RIC:-0.0960 prec@10:0.5266, SR:1.3491
BestTest performance: mse:0.0004 IC:0.0105 RIC:0.1522 prec@10:0.5287, SR:1.7961




Epochs :34, cur_loss : 0.00049,cur_reg_loss : 0.00048,cur_rank_loss : 0.00007:  92%|█████████▏| 680/740 [00:09<00:00, 72.78it/s] 


KeyboardInterrupt: 